# Two-group PPI: striatum–auditory cortex coupling by learning stage

ROI-to-ROI generalized PPI (`ppi_roi.py`): does coupling between the anterior caudate and auditory cortex depend on the task event, does it differ between learners (non-Mandarin, n = 12) and natives (Mandarin, n = 13), and does it change with learning?

**Model** (per participant × stage × seed; details in the `ppi_roi.py` docstring): the univariate `scrubbed-splitfb` design (sound, resp, fb_correct, fb_wrong, fb_noresp, drift, motion + aCompCor, FD/DVARS scrubbing), plus the seed time series (cleaned of drift and confounds, not task) and seed × condition interaction terms. There's no deconvolution (TR = 3 s). resp has no interaction term: it follows the sound by <1 s, so its term would nearly duplicate the sound term. Target time series are z-scored within run; runs are combined within stage by fixed effects.

**Seeds:** bilateral aCAU (main) and bilateral aPUT (comparison: the univariate feedback result is aCAU vs aPUT). **Targets:** the 14 auditory ROIs, and the three composites from the multivariate analyses (`multivariate_data.COMPOSITES`: Primary = HG, PT, PP; Association = STGa, STGp; IFG = pars opercularis, pars triangularis).

**Contrasts:**
- `seed`: task-independent coupling (sanity check)
- `ppifbcorrect`: coupling during correct feedback, both groups (**planned**)
- `ppisound`: coupling during the sound (+ response) period, both groups
- `ppifbcorrectvswrong`: learners only (natives have ~1 wrong trial per run)

**Planned tests:** aCAU seed, `ppifbcorrect`, three composites: learners − natives with stages averaged, and the learning-stage trend [−1, 0, 1] learners − natives; FDR across the three composites (`group_stats`, as in the univariate and multivariate notebooks). Everything else is secondary or exploratory.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join('..', '10_multivariate'))
import group_stats as gs
import multivariate_data as mvd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 24)

nilearn_dir = '/ix1/bchandrasekaran/krs228/data/FLT/data_denoised/derivatives/nilearn'
group_dir = os.path.join(nilearn_dir, 'group_space-T1w_fwhm-0.00')
out_dir = os.path.join(group_dir, 'stats_two-group', 'ppi')
os.makedirs(out_dir, exist_ok=True)

seeds = ('aCAU', 'aPUT')
contrasts = ('seed', 'ppisound', 'ppifbcorrect', 'ppifbwrong', 'ppifbcorrectvswrong')
composite_network = 'auditory-composite'
composites = list(mvd.COMPOSITES)

# resampling settings, matching robustness_checks.py
n_perms, n_boot, seed = 10000, 10000, 0
colors = {'non-Mandarin': '#2a6fdb', 'Mandarin': '#d9822b'}   # as in group_level_two_group.ipynb
labels = {'non-Mandarin': 'learners', 'Mandarin': 'natives'}

## 1. Load and check

In [ ]:
ppi = {s: mvd.add_composites(gs.load_roi_values(os.path.join(group_dir, f'roi-values_ppi-seed-{s}'),
                                                networks=('auditory',), contrasts=contrasts))
       for s in seeds}
qc = pd.concat([pd.read_csv(os.path.join(group_dir, f'roi-values_ppi-seed-{s}', 'ppi-qc.tsv'), sep='\t')
                for s in seeds], ignore_index=True)

for s in seeds:
    print(f'--- {s} seed: participants per contrast x group x stage')
    display(gs.coverage(ppi[s]))

In [ ]:
# per run: retained volumes, residual dof, and how collinear the interaction terms are
# (feedback is one TR after the sound, so ppi_sound and ppi_fb_* share variance)
vif_cols = [c for c in qc.columns if c.startswith('vif_')]
display(qc[qc.seed == 'aCAU'].groupby('participant_group')[['n_retained', 'residual_dof', 'seed_n_voxels']
                                                            + vif_cols].describe().T.round(1))
# natives' wrong-feedback trials per stage (why ppifbcorrectvswrong is learners only)
display(qc[qc.seed == 'aCAU'].groupby(['participant_group', 'learning_stage'])['n_fb_wrong'].describe()
        [['mean', 'min', 'max']].round(1))

## 2. Task-independent coupling (sanity check)

The seed term should be positive across auditory cortex in both groups: the ROIs share low-frequency fluctuation even after confounds. If it isn't, something upstream is wrong (masks, seed extraction), and the PPI terms can't be trusted.

In [ ]:
seed_coupling = gs.baseline_tests(ppi['aCAU'][ppi['aCAU'].contrast == 'seed'])
seed_coupling[['network', 'participant_group', 'region', 'n', 'mean', 't', 'p', 'q']].round(3)

## 3. Planned: aCAU coupling during correct feedback, learners vs natives

Composites, hemispheres and stages averaged per participant, then learners − natives (Welch t, d, group-label permutation p; FDR across the three composites). Then the learning-stage trend, and the group × stage ANOVA as a descriptive companion.

In [ ]:
acau = ppi['aCAU']
comp = acau[acau.network == composite_network]

# within group: is there feedback-related coupling at all?
within = gs.baseline_tests(comp[comp.contrast == 'ppifbcorrect'])
display(within[['participant_group', 'region', 'n', 'mean', 't', 'p', 'q', 'd']].round(3))

planned_level = gs.group_difference_tests(comp, contrasts=('ppifbcorrect',), n_perms=n_perms, seed=seed)
display(planned_level.round(3))

In [ ]:
trends = gs.subject_trends(acau)
comp_trends = trends[trends.network == composite_network]
planned_trend = gs.trend_difference_tests(comp_trends, contrasts=('ppifbcorrect',), n_perms=n_perms, seed=seed)
display(planned_trend.round(3))

# full battery (within-group trend vs 0, LOSO, bootstrap) per composite
battery = pd.concat([gs.trend_group_difference(comp_trends, 'ppifbcorrect', r, n_perms=n_perms, n_boot=n_boot, seed=seed)
                     for r in composites], ignore_index=True)
battery.round(3)

In [ ]:
anova = gs.group_by_stage_anova(comp, contrasts=('ppifbcorrect',))
anova.round(3)

## 4. Learners: correct vs wrong feedback

Learners only. Is aCAU–cortex coupling stronger after correct than wrong feedback, and does that difference change with learning? Trend tests are learners only (t-test, exact sign-flip, bootstrap CI; FDR across composites).

In [ ]:
learners = comp[(comp.contrast == 'ppifbcorrectvswrong') & (comp.participant_group == gs.GROUPS[0])]
display(gs.baseline_tests(learners)[['region', 'n', 'mean', 't', 'p', 'q', 'd']].round(3))
display(gs.baseline_tests(learners, by_stage=True)[['region', 'learning_stage', 'n', 'mean', 't', 'p', 'q']].round(3))

# learners' trend vs 0 (trend_group_difference needs both groups, and natives mostly lack this contrast)
from scipy import stats
learner_trends = comp_trends[(comp_trends.contrast == 'ppifbcorrectvswrong')
                             & (comp_trends.participant_group == gs.GROUPS[0])]
rows = []
for region in composites:
    v = learner_trends.loc[learner_trends.region == region, 'trend'].values
    t, p = stats.ttest_1samp(v, 0)
    boot = gs.bootstrap_ci(v, n_boot=n_boot, rng=np.random.default_rng(seed))
    rows.append({'region': region, 'n': len(v), 'mean': v.mean(), 't': t, 'p': p,
                 'permutation_p': gs.exact_sign_flip_permutation_test(v)['p_value'],
                 'ci_lower': boot['ci_lower'], 'ci_upper': boot['ci_upper']})
cvw_trend = pd.DataFrame(rows).assign(q=lambda d: gs.fdr(d.p))
cvw_trend.round(3)

## 5. Coupling during the sound

The stimulus-locked counterpart: does aCAU couple with auditory cortex during tone processing, and does that differ by group or change with learning?

In [ ]:
display(gs.baseline_tests(comp[comp.contrast == 'ppisound'])[['participant_group', 'region', 'n', 'mean', 't', 'p', 'q']].round(3))
display(gs.group_difference_tests(comp, contrasts=('ppisound',), n_perms=n_perms, seed=seed).round(3))
gs.trend_difference_tests(comp_trends, contrasts=('ppisound',), n_perms=n_perms, seed=seed).round(3)

## 6. Seed specificity: aPUT, and aCAU − aPUT

The univariate feedback result is a caudate-vs-putamen difference, so the comparison seed is aPUT. An aCAU-specific coupling effect should show up in the per-participant aCAU − aPUT difference, not just in aCAU.

In [ ]:
aput_comp = ppi['aPUT'][ppi['aPUT'].network == composite_network]
display(gs.group_difference_tests(aput_comp, contrasts=('ppifbcorrect', 'ppisound'), n_perms=n_perms, seed=seed).round(3))

keys = ['participant_id', 'participant_group', 'network', 'contrast', 'region', 'region_hemi', 'hemisphere',
        'learning_stage']
seed_diff = (ppi['aCAU'].merge(ppi['aPUT'], on=keys, suffixes=('_aCAU', '_aPUT'))
             .assign(beta=lambda d: d.beta_aCAU - d.beta_aPUT)[keys + ['beta']])
diff_comp = seed_diff[seed_diff.network == composite_network]
display(gs.baseline_tests(diff_comp[diff_comp.contrast.isin(['ppifbcorrect', 'ppisound'])])
        [['contrast', 'participant_group', 'region', 'n', 'mean', 't', 'p', 'q']].round(3))
display(gs.group_difference_tests(diff_comp, contrasts=('ppifbcorrect', 'ppisound'), n_perms=n_perms, seed=seed).round(3))
gs.trend_difference_tests(gs.subject_trends(diff_comp), contrasts=('ppifbcorrect', 'ppisound'),
                          n_perms=n_perms, seed=seed).round(3)

## 7. Exploratory: per region

The 7 auditory regions (hemispheres averaged), FDR across regions within contrast. Not part of the planned family.

In [ ]:
regions = acau[acau.network == 'auditory']
region_level = gs.group_difference_tests(regions, contrasts=('ppifbcorrect', 'ppisound'), n_perms=n_perms, seed=seed)
region_trend = gs.trend_difference_tests(trends[trends.network == 'auditory'], contrasts=('ppifbcorrect', 'ppisound'),
                                         n_perms=n_perms, seed=seed)
display(region_level.round(3))
region_trend.round(3)

## 8. Figure: aCAU coupling by stage

In [ ]:
bilat = gs.bilateral_mean(comp)
fig, axes = plt.subplots(2, len(composites), figsize=(3.2 * len(composites), 5.6), sharey='row')
for row, contrast in zip(axes, ['ppifbcorrect', 'ppisound']):
    for ax, region in zip(row, composites):
        sub = bilat[(bilat.contrast == contrast) & (bilat.region == region)]
        for offset, group in zip((-0.05, 0.05), gs.GROUPS):
            stats_ = (sub[sub.participant_group == group].groupby('learning_stage').beta
                      .agg(['mean', 'sem']).reindex(gs.STAGE_ORDER))
            ax.errorbar(np.arange(3) + offset, stats_['mean'], yerr=stats_['sem'], marker='o', capsize=3,
                        color=colors[group], label=labels[group])
        ax.axhline(0, color='0.6', lw=0.8)
        ax.set_xticks(range(3), [s.capitalize() for s in gs.STAGE_ORDER])
        ax.set_title(f'{region}: {contrast}', fontsize=9)
    row[0].set_ylabel('aCAU coupling (PPI β)')
axes[0, 0].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(out_dir, 'ppi_seed-aCAU_composites_by-stage.png'), dpi=200)

## 9. Save tables

In [ ]:
for name, table in {'planned_level': planned_level, 'planned_trend': planned_trend, 'planned_battery': battery,
                    'group_by_stage_anova': anova, 'seed_coupling': seed_coupling,
                    'learners_correct-vs-wrong_trend': cvw_trend,
                    'exploratory_region_level': region_level, 'exploratory_region_trend': region_trend}.items():
    table.to_csv(os.path.join(out_dir, f'ppi_seed-aCAU_{name}.tsv'), sep='\t', index=False)
print('saved to', out_dir)